# RaCo + Kornia on Snowflake Container Runtime

This GPU notebook uses synthetic in-memory imagery and a separately staged, reviewed RaCo checkpoint. RaCo returns keypoints, detection/ranker scores and 2x2 covariance; Kornia applies a known transform. This is a repeatability illustration, not a learned matching or homography benchmark. Place `vision.py` in this notebook's Workspace folder first.

In [ ]:
from snowflake.snowpark.context import get_active_session

# Approved stage inventory; do not download arbitrary remote files.
staged_weights = get_active_session().sql("LIST @RACO_RLM_DEMO.PUBLIC.MODEL_ASSETS PATTERN='.*raco[.]pth'").collect()

In [ ]:
%pip install kornia==0.9.2 "numpy==1.26.4" "opencv-python<4.11"
# RaCo pinned to a reviewed commit of https://github.com/cvg/RaCo; --no-deps keeps the runtime's numpy/torch
%pip install --no-deps git+https://github.com/cvg/RaCo.git@ee06fa68dbb20303f334fbc257dad71d4149603b

In [ ]:
%%sql -r dataframe_1
PUT file:///path/to/raco.pth @RACO_RLM_DEMO.PUBLIC.MODEL_ASSETS AUTO_COMPRESS=FALSE OVERWRITE=TRUE;

In [ ]:
import raco, pathlib
pkg = pathlib.Path(raco.__file__).parent
print(pkg)
print(list(pkg.rglob('*.pt*')))

In [ ]:
import torch
import kornia
from raco import RaCo
from vision import covariance_is_valid, rotation_repeatability

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, 'kornia', kornia.__version__)
session = get_active_session()
if len(staged_weights) != 1:
    raise ValueError('Stage one approved raco.pth checkpoint before continuing')
CHECKPOINT_STAGE_PATH = '@RACO_RLM_DEMO.PUBLIC.MODEL_ASSETS/raco.pth'

In [ ]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as model_dir:
    downloaded = session.file.get(CHECKPOINT_STAGE_PATH, model_dir)
    if len(downloaded) != 1:
        raise ValueError('Expected exactly one approved RaCo checkpoint')
    weight_path = Path(model_dir) / 'raco.pth'
    if not weight_path.is_file():
        raise FileNotFoundError('Stage must contain raco.pth (uncompressed)')
    # A PyTorch checkpoint can execute code when loaded by an unsafe loader.
    # Only use a reviewed checkpoint supplied by your organization.
    extractor = RaCo(weights=str(weight_path)).eval().to(DEVICE)
    yy, xx = torch.meshgrid(torch.arange(256), torch.arange(256), indexing='ij')
    checker = (((xx // 24 + yy // 24) % 2) * 0.6 + 0.2).float()
    checker[35:215, 102:109] = 1.0
    checker[123:132, 30:200] = 0.0
    image = checker[None, None].repeat(1, 3, 1, 1).to(DEVICE)
    rotated = kornia.geometry.transform.rotate(image, torch.tensor([90.], device=image.device))
    # Compare within a radius: Kornia resamples about the image center.
    with torch.no_grad():
        original_output = extractor.extract(image)
        rotated_output = extractor.extract(rotated)

In [ ]:
import json
import uuid
import matplotlib.pyplot as plt

points = original_output['keypoints'][0].detach().cpu()
rotated_points = rotated_output['keypoints'][0].detach().cpu()
ranks = original_output['ranker_scores'][0].detach().cpu()
indices = torch.argsort(ranks, descending=True)[:min(64, len(points))].tolist()
covs = original_output['covariances'][0].detach().cpu()
assert all(covariance_is_valid(covs[index].tolist()) for index in indices)
repeatability = rotation_repeatability(points[indices].tolist(), rotated_points.tolist(), 256, 256, radius=6)
run_id = str(uuid.uuid4())
records = [(run_id, int(index), json.dumps({'x': float(points[index, 0]), 'y': float(points[index, 1]), 'ranker_score': float(ranks[index]), 'detection_score': float(original_output['keypoint_scores'][0, index]), 'covariance': covs[index].tolist(), 'repeatability_at_6px': repeatability})) for index in indices]
for run, point_index, payload in records:
    session.sql('INSERT INTO RACO_RLM_DEMO.PUBLIC.VISION_KEYPOINTS (RUN_ID, POINT_INDEX, PAYLOAD) SELECT ?, ?, PARSE_JSON(?)', params=[run, point_index, payload]).collect()
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image[0].permute(1, 2, 0).cpu()); axes[0].scatter(points[indices, 0], points[indices, 1], s=10)
axes[1].imshow(rotated[0].permute(1, 2, 0).cpu()); axes[1].scatter(rotated_points[:, 0], rotated_points[:, 1], s=4)
for axis in axes: axis.set_axis_off()
plt.show()
print({'run_id': run_id, 'keypoints_saved': len(records), 'repeatability_at_6px': repeatability})